This notebook has the goal to check and promote pertinent corrections in the tables

# Imports

In [12]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructField, StringType, IntegerType, FloatType, DoubleType, DateType, StructType
from pyspark.sql.window import Window
from pyspark.sql.session import SparkSession

# Auxiliary Functions

In [2]:
def null_or_blank(df):
  null_or_blank = df.select(
                          [
                            F.count(
                                     F.when(
                                             F.col(c).isNull() 
                                             | (F.col(c).cast('string') == '') 
                                             | (F.col(c).cast('string').rlike('^\\s*$')),
                                             c
                                           )
                                   ).alias(c)
                            for c in df.columns
                          ]
                         )
  return null_or_blank

In [3]:
def check_duplicates(df, column):
  columns = column if isinstance(column, (list, tuple)) else [column]
  return df.groupBy(*columns).count().filter(F.col('count') > 1).show()

In [4]:
def check_and_remove_duplicates(df, column):
  columns = column if isinstance(column, (list, tuple)) else [column]
  check_duplicates(df, column)
  duplicates = df.groupBy(*columns).count().filter(F.col('count') > 1)
  if duplicates.count() > 0:
    df = df.dropDuplicates(columns)
    print('Duplicates removed.')
  else:
    print('No duplicates found.')
  return df

## Table: bandeiras_tarifarias -> tax_flags

In [5]:
TABLE_ORIGIN = 'bandeiras_tarifarias'

TABLE_DESTINY = 'tax_flags'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
             .withColumnsRenamed(
                                  {
                                    'mes_referencia' : 'month',
                                    'bandeira' : 'flag',
                                    'adicional_rs_kwh' : 'additional_charge_kwh',
                                  }
                                )
             .withColumn('additional_charge_kwh', F.col('additional_charge_kwh').cast('float'))
     )



In [6]:
df.printSchema()

root
 |-- month: string (nullable = true)
 |-- flag: string (nullable = true)
 |-- additional_charge_kwh: float (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_path: string (nullable = true)
 |-- _ingestion_ts: timestamp (nullable = true)



In [7]:
check_duplicates(df,'month')


+-----+-----+
|month|count|
+-----+-----+
+-----+-----+



In [8]:
display(null_or_blank(df))

,month,flag,additional_charge_kwh,_source_file,_source_path,_ingestion_ts
0,0,0,0,0,0,0


In [9]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{TABLE_DESTINY}")

In [10]:
df = spark.read.table(f"silver.{TABLE_DESTINY}")

display(df)

,month,flag,additional_charge_kwh,_source_file,_source_path,_ingestion_ts
0,2025-10,VERMELHA_P1,0.04463,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
1,2025-11,VERMELHA_P1,0.04463,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
2,2025-12,AMARELA,0.01885,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
3,2026-01,VERDE,0.00000,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
4,2026-02,VERDE,0.00000,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
5,2026-03,VERDE,0.00000,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
6,2026-04,VERDE,0.00000,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
7,2026-05,AMARELA,0.01885,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
8,2026-06,AMARELA,0.01885,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541
9,2026-07,AMARELA,0.01885,bandeiras_tarifarias.csv,dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/bandeiras_tarifarias.csv,2026-10-15 23:05:31.907541


## Table: clientes_crm -> customers_crm

In [0]:
TABLE_ORIGIN = 'clientes_crm'

TABLE_DESTINY = 'customers_crm'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
             .withColumnsRenamed(
                                  {
                                    'id_cliente' : 'customer_id',
                                    'razao_social' : 'legal_name',
                                    'cnpj' : 'tax_id',
                                    'segmento' : 'segment',
                                    'uf_sede' : 'uf',
                                    'email_financeiro' : 'email',
                                    'data_cadastro' : 'onboarding_date',
                                    'atualizado_em' : 'updated_in'
                                  }
                                )
     )

In [0]:
df = (
        df.withColumn('tax_id', F.regexp_replace(F.col('tax_id'), r'[^0-9]', ''))
          .withColumn(
                       'onboarding_date', 
                       F.date_format(
                                      F.to_date(
                                                 F.regexp_replace(F.col('onboarding_date'), r'(\d{2})/(\d{2})/(\d{4})', '$3-$2-$1'),'yyyy-MM-dd'
                                               ), 
                                      'yyyy-MM-dd')
                     )
          .withColumn('updated_in', F.date_format(F.col('updated_in').cast('string'), 'yyyy-MM-dd'))
          .withColumn('uf', F.upper(F.col('uf')))
     )

display(df)

Let's check if 'tax_id' has only values with 14 characters as length (brazillian pattern):

In [0]:
df.groupBy(F.length(F.col('tax_id'))).count().show()

Checking the UF's values (e.g whether are real regions):

In [0]:
df.groupBy('uf').count().show()

Checking potential duplicates in customers:

In [0]:
check_duplicates(df, 'customer_id')

According that discussed with stakeholders: in case of customers' duplicity we must maintain the most recent 'updated_in'

In [0]:
df = (
        df.withColumn(
                       'rn', 
                       F.row_number().over(Window.partitionBy('customer_id').orderBy(F.desc('updated_in')))
                     )
          .filter(F.col('rn') == 1)
          .drop('rn')
    )

Checking null's and blank's values:

In [0]:
display(null_or_blank(df))

The 'email' and 'segment' could be important to send charges and conduct geographic analysis, respectively. However, the main goal here not seem impacted. But regardless it, we'll sinalize this to CS team to provide adequated corrections.

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{TABLE_DESTINY}")

## Table: contratos_uc -> contracts_uc

In [0]:
spark.read.table('bronze.contratos_uc').show(5)

In [0]:
spark.read.table('bronze.contratos_uc').printSchema()

In [0]:
TABLE_ORIGIN = 'contratos_uc'

TABLE_DESTINY = 'contracts_uc'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
             .withColumnsRenamed(
                                  {
                                    'id_contrato' : 'contract_id',
                                    'id_cliente' : 'customer_id',
                                    'distribuidora' : 'distributor',
                                    'id_usina' : 'station_id',
                                    'subgrupo' : 'subgroup',
                                    'cota_kwh_mes' : 'month_limit_kwh',
                                    'desconto' : 'discount_pct',
                                    'data_inicio' : 'start_date',
                                    'data_fim' : 'end_date',
                                  }
                                )
             .withColumn('distributor', F.upper(F.col('distributor')))
             .withColumn('uf', F.upper(F.col('uf')))
             .withColumn('regional', F.upper(F.col('regional')))
             .withColumn(
                          'start_date', 
                          F.date_format(
                                         F.to_date(
                                                    F.regexp_replace(F.col('start_date'), r'(\d{2})/(\d{2})/(\d{4})', '$3-$2-$1'),
                                                    'yyyy-MM-dd'
                                                  ), 
                                         'yyyy-MM-dd')
                        )
             .withColumn(
                          'end_date', 
                          F.date_format(
                                         F.to_date(
                                                    F.regexp_replace(F.col('end_date'), r'(\d{2})/(\d{2})/(\d{4})', '$3-$2-$1'),
                                                    'yyyy-MM-dd'
                                                  ), 
                                         'yyyy-MM-dd')
                        )
             .withColumn('discount_pct', F.regexp_replace(F.regexp_replace(F.col('discount_pct'), '%', ''), ',', '.').cast('decimal(5,2)'))
             .withColumn('discount_pct', F.when(F.col('discount_pct') >= 1, F.col('discount_pct') / 100).otherwise(F.col('discount_pct')))
             .withColumn('status', F.upper(F.col('status')))
             .withColumn('month_limit_kwh', F.col('month_limit_kwh').cast('float'))
             
     )

display(df)

In [0]:
display(null_or_blank(df))

'end_date' null means the contract is still active. No problem.

In [0]:
display(check_duplicates(df,'contract_id'))

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{TABLE_DESTINY}")

## Table: faturamento_erp_202510_202609 -> invoice_erp_202510_to_202609

In [0]:
spark.table('bronze.faturamento_erp_202510_202609').show(10)

In [0]:
spark.table('bronze.faturamento_erp_202510_202609').printSchema()

In [0]:
TABLE_ORIGIN = 'faturamento_erp_202510_202609'

TABLE_DESTINY = 'invoice_erp_202510_to_202609'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
                  .withColumnsRenamed(
                                       {
                                         'num_fatura' : 'invoice_id',
                                         'id_cliente' : 'customer_id',
                                         'mes_referencia' : 'month',
                                         'data_emissao' : 'invoice_date',
                                         'kwh_faturado' : 'kwh_invoiced',
                                         'tarifa_cheia_aplicada' : 'full_charge_applied',
                                         'desconto_aplicado' : 'discount_applied',
                                         'valor_fio_b_deduzido' : 'wire_b_deducted',
                                         'valor_faturado' : 'invoiced_value',
                                       }
                                     )
                  .withColumn('kwh_invoiced', F.col('kwh_invoiced').cast('bigint'))
                  .withColumn('full_charge_applied', F.col('full_charge_applied').cast('float'))
                  .withColumn('discount_applied', F.col('discount_applied').cast('float'))
                  .withColumn('wire_b_deducted', F.col('wire_b_deducted').cast('float'))
                  .withColumn('invoiced_value', F.col('invoiced_value').cast('float'))
     )

display(df)

In [0]:
display(null_or_blank(df))

In [0]:
display(check_duplicates(df,'invoice_id'))

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{TABLE_DESTINY}")

## Tables: "medicoes..."

We have tables for each (regional, month). The idea is combine all regional's mensuration in a unique file per month after promote corrections

Business rules to help handling with potential data problems into measurements' tables:

- Every "consumer unity" (column uc) must have 10 digits. If necessary, complete with 0's left;
- Energy measurements must be in kWh;
- Primary key in the measurements file are (uc, month of energy lecture). We shouldn't have duplicates in the primary key;
- If we have any file name ending with any version, this is mandatory. For example if we have "medicoes_co_202510" and "medicoes_co_202510_v2" we need to delete "medicoes_co_202510" and keep "medicoes_co_202510_v2";

In [0]:
list_regional = ['co', 'nordeste', 'sul', 'sudeste']

list_months = sorted(
                        set(
                               row.tableName.split('_')[2]
                               for row in spark.sql("SHOW TABLES IN bronze").collect()
                               if row.tableName.startswith('medicoes_')
                           )
                    )

month_map = F.create_map(
                          F.lit('jan'), F.lit('01'), F.lit('fev'), F.lit('02'),
                          F.lit('mar'), F.lit('03'), F.lit('abr'), F.lit('04'),
                          F.lit('mai'), F.lit('05'), F.lit('jun'), F.lit('06'),
                          F.lit('jul'), F.lit('07'), F.lit('ago'), F.lit('08'),
                          F.lit('set'), F.lit('09'), F.lit('out'), F.lit('10'),
                          F.lit('nov'), F.lit('11'), F.lit('dez'), F.lit('12'),
                        )

In [0]:
def _extract_version(t):
    """Extract version number from a versioned table name (e.g., medicoes_co_202510_v2 → 2)."""
    parts = t.rsplit("_v", 1)
    return int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0


for m in list_months:  
    df_month = None
    for reg in list_regional:
        base_name = f"medicoes_{reg}_{m}"

        # Find all matching tables, including versioned variants (e.g., _v2, _v3)
        all_bronze_tables = [r.tableName for r in spark.sql("SHOW TABLES IN bronze").collect()]
        matching = [t for t in all_bronze_tables if t == base_name or t.startswith(base_name + "_v")]
        if not matching:
            print(f"bronze.{base_name} not found — skipping")
            continue

        # Keeping latest versioned (if exists)
        versioned = [t for t in matching if t != base_name]
        if versioned:
            table_origin = sorted(versioned, key=_extract_version, reverse=True)[0]
            if base_name in matching:
                print(f"{base_name} superseded by {table_origin}")
        else:
            table_origin = base_name

        # Reading table
        df_bronze = spark.read.table(f"bronze.{table_origin}")

        if 'Unidade_Consumidora' in df_bronze.columns:
            # ---- CSV format (regional: nordeste) ----
            df = (
                  df_bronze.withColumnsRenamed(
                                                {
                                                  'Unidade_Consumidora': 'uc',
                                                  'Mês_Referência': 'generated_at',
                                                  'Data_Leitura': 'reading_at',
                                                  'Consumo__MWh_': 'consume_mwh',
                                                  'Energia_Compensada__MWh_': 'compensated_energy_mwh',
                                                  'Saldo_Créditos__MWh_': 'balance_mwh',
                                                }
                                              )
                           .withColumn(
                                        'generated_at',
                                        F.date_format(
                                                       F.to_date(
                                                                  F.concat(
                                                                            F.lit('20'),
                                                                            F.element_at(F.split(F.col('generated_at'), '/'), 2),
                                                                            F.lit('-'),
                                                                            month_map[F.element_at(F.split(F.col('generated_at'), '/'), 1)],
                                                                            F.lit('-01')
                                                                          ),
                                                                  'yyyy-MM-dd'
                                                                ),
                                                       'yyyy-MM-dd'
                                                     )
                                      )
                           .withColumn('reading_at', F.date_format(F.to_date(F.col('reading_at'), 'dd/MM/yyyy'), 'yyyy-MM-dd'))
                           .withColumn(
                                        'consume_kwh',
                                        F.regexp_replace(F.col('consume_mwh'), ',', '.').cast('double') * F.lit(1000)
                                      )
                           .withColumn(
                                        'compensated_energy_kwh',
                                        F.regexp_replace(F.col('compensated_energy_mwh'), ',', '.').cast('double') * F.lit(1000)
                                      )
                           .withColumn(
                                        'balance_kwh',
                                        F.regexp_replace(F.col('balance_mwh'), ',', '.').cast('double') * F.lit(1000)
                                      )
                           .withColumn('regional', F.lit(reg))
                           .select(
                                    'generated_at',
                                    'reading_at',
                                    'regional',
                                    'uc',
                                    'consume_kwh',
                                    'compensated_energy_kwh',
                                    'balance_kwh',
                                    '_source_file',
                                    '_source_path',
                                    '_ingestion_ts'
                                  )
                  )
        elif 'unidade_consumidora' in df_bronze.columns:
            # ---- CSV format (regional: sul) ----
            df = (
                    df_bronze.withColumnsRenamed(
                                                  {
                                                    'unidade_consumidora': 'uc',
                                                    'mes_referencia': 'generated_at',
                                                    'data_leitura': 'reading_at',
                                                    'consumo_kwh': 'consume_kwh',
                                                    'compensado_kwh': 'compensated_energy_kwh',
                                                    'saldo_kwh': 'balance_kwh',
                                                  }
                                                )
                             .withColumn(
                                          'generated_at',
                                          F.date_format(
                                                        F.to_date(
                                                                  F.concat(
                                                                            F.col('generated_at'),
                                                                            F.lit('-01')
                                                                          ),
                                                                  'yyyy-MM-dd'
                                                                 ),
                                                        'yyyy-MM-dd'
                                                       )
                                        )
                             .withColumn('reading_at', F.date_format(F.to_date(F.col('reading_at')), 'yyyy-MM-dd'))
                             .withColumn('consume_kwh', F.col('consume_kwh').cast('double'))
                             .withColumn('compensated_energy_kwh', F.col('compensated_energy_kwh').cast('double'))
                             .withColumn('balance_kwh', F.col('balance_kwh').cast('double'))
                             .withColumn('regional', F.lit(reg))
                             .select(
                                      'generated_at',
                                      'reading_at',
                                      'regional',
                                      'uc',
                                      'consume_kwh',
                                      'compensated_energy_kwh',
                                      'balance_kwh',
                                      '_source_file',
                                      '_source_path',
                                      '_ingestion_ts'
                                    )
                 )
        elif 'UC' in df_bronze.columns:
            # ---- CSV format (regional: sudeste) ----
            df = (
                    df_bronze.withColumnsRenamed(
                                                  {
                                                    'UC': 'uc',
                                                    'MES_REF': 'generated_at',
                                                    'DT_LEITURA': 'reading_at',
                                                    'CONSUMO_KWH': 'consume_kwh',
                                                    'ENERGIA_COMPENSADA_KWH': 'compensated_energy_kwh',
                                                    'SALDO_CREDITOS_KWH': 'balance_kwh',
                                                  }
                                                )
                             .withColumn(
                                          'generated_at',
                                          F.date_format(
                                                         F.to_date(
                                                                    F.concat(
                                                                              F.element_at(F.split(F.col('generated_at'), '/'), 2),
                                                                              F.lit('-'),
                                                                              F.element_at(F.split(F.col('generated_at'), '/'), 1),
                                                                              F.lit('-01')
                                                                            ),
                                                                    'yyyy-MM-dd'
                                                                  ),
                                                         'yyyy-MM-dd'
                                                       )
                                        )
                             .withColumn(
                                          'reading_at',
                                          F.date_format(
                                                        F.coalesce(
                                                                    F.try_to_date(F.col('reading_at'), 'dd/MM/yyyy'),
                                                                    F.try_to_date(F.col('reading_at'), 'yyyy-MM-dd')
                                                                  ),
                                                        'yyyy-MM-dd'
                                                       )
                                        )
                             .withColumn('consume_kwh', F.col('consume_kwh').cast('double'))
                             .withColumn('compensated_energy_kwh', F.col('compensated_energy_kwh').cast('double'))
                             .withColumn('balance_kwh', F.col('balance_kwh').cast('double'))
                             .withColumn('regional', F.lit(reg))
                             .select(
                                      'generated_at',
                                      'reading_at',
                                      'regional',
                                      'uc',
                                      'consume_kwh',
                                      'compensated_energy_kwh',
                                      'balance_kwh',
                                      '_source_file',
                                      '_source_path',
                                      '_ingestion_ts'
                                    )
                 )
        else:
            # ---- JSON format (regional: co) ----
            df = (
                    df_bronze.withColumnsRenamed({'gerado_em': 'generated_at'})
                             .withColumn('generated_at', F.date_format(F.to_date(F.col('generated_at')), 'yyyy-MM-dd'))
                             .withColumn('m', F.explode('medicoes'))
                             .drop('medicoes', 'origem', 'regional', 'referencia')
                             .withColumn('regional', F.lit(reg))
                             .select(
                                      'generated_at',
                                      'regional',
                                      F.date_format(F.to_date(F.col('m.leitura')), 'yyyy-MM-dd').alias('reading_at'),
                                      F.col('m.uc').alias('uc'),
                                      F.col('m.valores.consumo').alias('consume_kwh'),
                                      F.col('m.valores.energia_compensada').alias('compensated_energy_kwh'),
                                      F.col('m.valores.saldo').alias('balance_kwh'),
                                      '_source_file',
                                      '_source_path',
                                      '_ingestion_ts'
                                    )
                 )

        if df_month is None:
            df_month = df
        else:
            df_month = df_month.unionByName(df)

        print(f"✓ {table_origin}")

    if df_month is not None:
        # uc must have 10 digits — pad with leading zeros
        df_month = df_month.withColumn('uc', F.lpad(F.trim(F.col('uc').cast('string')), 10, '0'))

        # primary key is (uc, generated_at) — remove duplicates
        df_month = check_and_remove_duplicates(df_month, ['uc', 'generated_at'])

        table_destiny = f"silver.measurements_{m}"
        df_month.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_destiny)
        print(f"\n✓ {table_destiny} written ({df_month.count()} rows)\n")


In [0]:
# Delete all tables in the 'silver' schema
silver_tables = spark.sql("SHOW TABLES IN silver").collect()

if not silver_tables:
    print("No tables found in silver schema.")
else:
    for row in silver_tables:
        table_name = f"silver.{row.tableName}"
        print(f"Dropping {table_name} ...")
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    print(f"\n✓ Dropped {len(silver_tables)} table(s) from silver schema.")

## Table: parametros_lei_14300 -> law_14300_parameters

In [0]:
spark.table('bronze.parametros_lei_14300').show(truncate=False)

In [0]:
TABLE_ORIGIN = 'parametros_lei_14300'

TABLE_DESTINY = 'law_14300_parameters'

df = (
        spark.table(f"bronze.{TABLE_ORIGIN}")
             .filter(F.col('ano') != 'corte_gd1')
             .withColumnsRenamed({
                                   'ano' : 'year_start_contract',
                                   'percentual_fio_b_nao_compensado' : 'pct_b_wire_not_compensated'
                                })
             .withColumn('pct_b_wire_not_compensated', F.col('pct_b_wire_not_compensated').cast('float'))
     )

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{TABLE_DESTINY}")

## Tables: "tarifas..."

In [15]:
spark.table('bronze.tarifas_sudeste').show(truncate=False)

+---------------------------+---+---------------+------------+---------+-----------+-----------------+----+----------+-------------------+---------------------------------------------------------------------------------------+--------------------------+
|DISTRIBUIDORA              |UF |INICIO_VIGENCIA|FIM_VIGENCIA|TE_RS_KWH|TUSD_RS_KWH|TUSD_FIO_B_RS_KWH|ICMS|PIS_COFINS|_source_file       |_source_path                                                                           |_ingestion_ts             |
+---------------------------+---+---------------+------------+---------+-----------+-----------------+----+----------+-------------------+---------------------------------------------------------------------------------------+--------------------------+
|Cemig Distribuicao         |MG |28/05/2025     |27/05/2026  |0,30100  |0,47200    |0,23500          |18% |0,0520    |tarifas_sudeste.csv|dbfs:/Volumes/workspace/landing/general_files/edg_data/02_premissas/tarifas_sudeste.csv|2026-10-15 2

In [ ]:
list_regional = ['centro_oeste', 'nordeste', 'sul', 'sudeste']

In [ ]:
for reg in list_regional:
    base_name = f"tarifas_{reg}_{m}"
    
    # Find all matching tables, including versioned variants (e.g., _v2, _v3)
    all_bronze_tables = [r.tableName for r in spark.sql("SHOW TABLES IN bronze").collect()]
    matching = [t for t in all_bronze_tables if t == base_name or t.startswith(base_name)]
    if not matching:
        print(f"bronze.{base_name} not found — skipping")
        continue

    # Reading table
    df_bronze = spark.read.table(f"bronze.{table_origin}")

    df  = (
            df_bronze.withColumnsRenamed({
                                            'atualizado_em' : 'updated_at',
                                            'responsavel' : 'sponsor',
                                            'tarifas'
                                         })
          )

    if df_tarifa is None:
        df_tarifa = df
    else:
        df_tarifa = df_tarifa.unionByName(df)
    
        print(f"✓ {table_origin}")